# 📝 리랭킹 과제 LV1(기초)

교안 01의 리랭킹을 한 단계씩 연습합니다. 가상 회사의 IT 헬프데스크 도움말 41개(`lv1_docs.json`)와 평가 질문 7개(`lv1_questions.json`)를 씁니다. VPN·계정·메일·장비처럼 같은 낱말을 쓰는 도움말이 여러 개라서 1차 검색 후보에 비슷하지만 답이 아닌 문서가 섞입니다. 도움말과 운영 규칙은 모두 수업용 창작입니다.

- 1번: 하이브리드 검색으로 후보를 한 번 준비하고 최종 개수와 구분
- 2~3번: Cross-Encoder로 질문·후보 쌍의 점수를 구하고 공식 리랭커로 재정렬
- 4번: 후보를 줄였을 때 리랭킹으로 찾을 수 없는 근거 확인(비교 표 제공)
- 5번: ColBERT 방식(BGE-M3)으로 같은 후보를 재정렬
- 6번: 검색과 리랭킹을 하나의 검색기로 연결
- 7번: 모델만 바꿔 평가 질문 5개의 같은 후보로 비교(비교 표·저장 제공)
- 8번: 운영 조건에 맞는 리랭커 선택(서술)

`.env`의 OpenAI 키가 필요합니다. 하이브리드 검색의 Dense 쪽 임베딩만 요청하며(문서 41개와 검색 질문 8회, 1번 자가채점의 확인 검색 포함) GPT는 호출하지 않습니다. 리랭커 세 모델(Qwen3-Reranker-0.6B, BGE-M3, BGE-reranker-v2-m3)은 로컬 CPU에서 실행하며 교안 01의 다운로드 셀로 받은 가중치를 그대로 씁니다. Qwen은 후보 한 묶음을 처리하는 데 수 초 이상 걸릴 수 있습니다. 세 모델을 한 커널에 올리므로 교안 01 커널은 종료하고 시작하세요. 7번 앞에서는 5번 뒤로 쓰지 않는 ColBERT 모델을 메모리에서 내립니다.

<strong>풀이 방법</strong>: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 핵심 호출 코드를 작성합니다. `...`는 호출식 전체나 여러 줄의 코드로 바꿀 수 있습니다. 질문 준비·결과 표·비교·출력·저장 코드는 완성되어 있습니다. 문항별 핵심 동작만 자가채점하며, 리랭킹 순위·점수·처리 시간은 고정하지 않습니다. 앞 문항의 변수를 다음 문항에서 이어 씁니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하세요. 경로·JSON 함수·원문 ID는 지난 교안의 방식을 이어갑니다.

아래 준비 코드는 한 셀입니다. `# ====` 구분선으로 역할을 나눴습니다. 위에서부터 실행하면 도움말 문서와 평가 질문을 읽고, 지난 단원의 하이브리드 검색기 `hybrid`(BM25·Dense 각 5개), 교안 01과 같은 리랭커 모델 `cross_encoder`(Qwen3-Reranker-0.6B), 전후 순위 대조 함수 `rank_rows`, 근거 원문의 순위를 찾는 함수 `evidence_rank`가 준비됩니다. 이 셀을 다시 실행하면 문서를 다시 임베딩합니다.


In [ ]:
# ====================================================================
# 1) 라이브러리 가져오기
# 문서·표·토큰화·검색에 사용할 라이브러리를 가져옵니다.
# ====================================================================
# 본문과 출처를 같은 Document에 보관합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# ====================================================================
# 2) 경로와 JSON 입출력
# material_dir를 기준으로 데이터를 읽고 결과를 저장하는 함수를 준비합니다.
# ====================================================================
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

# ====================================================================
# 3) 환경변수와 임베딩 모델
# .env의 API 키를 읽고 문서와 질문에 공통으로 사용할 임베딩 모델을 설정합니다.
# ====================================================================
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 아래 적재·검색 구간에서 요청합니다.")

# ====================================================================
# 4) 원문을 Document로 변환하는 함수
# 본문과 원문 ID·출처·분류를 함께 보관합니다.
# ====================================================================
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 유지하고 필터 필드만 추가합니다.
    documents = []
    for record in records:
        document = Document(
            id=record["doc_id"],
            page_content=record["text"],
            metadata={"source_id": record["doc_id"], "title": record["title"],
                      "url": record["url"], **record["metadata"]},
        )
        documents.append(document)
    return documents

# ====================================================================
# 5) 과제 데이터 읽기
# 헬프데스크 도움말 41개를 읽고 Document 목록으로 바꿉니다.
# ====================================================================
# 데이터 구조는 앞 5행으로 확인하고, 검색에는 전체 문서를 사용합니다.
records = read_json("lv1_docs.json")
print("전체 문서 수:", len(records))
display(pd.DataFrame(records).head())
documents = make_documents(records)

# ====================================================================
# 6) 결과 출력과 한국어 토큰화
# show_results는 원문을 print로 보여 주고, kiwi_tokenize는 BM25에 쓸 토큰을 만듭니다.
# ====================================================================
def show_results(documents):
    """표시 순서와 원문 ID·메타데이터·본문 전체를 보여 줍니다."""
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    print("결과 문서 수:", len(documents))
    for order, doc in enumerate(documents, start=1):
        print(f"[{order}] 원문 ID:", doc.metadata["source_id"])
        print("메타데이터:", doc.metadata)
        # 본문은 별도 줄에 출력해 원문의 줄바꿈을 그대로 읽습니다.
        print("본문:")
        print(doc.page_content)
        print()

# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF에서 온 반각 가운뎃점(･)은 분석기가 앞뒤를 다른 낱말로 끊으므로 가운뎃점(·)으로 바꿉니다.
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    words = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith("N") or token.tag in {"SL", "SN"}:
            words.append(token.form.lower())
    return words

# ====================================================================
# 7) 도움말 분류와 평가 질문
# 분류별 문서 수를 보고, 사람이 원문을 읽고 정한 질문과 근거 원문 ID를 읽습니다.
# ====================================================================
# 분류별 도움말 수를 보고 같은 주제의 문서가 여러 개인지 확인합니다.
display(pd.Series([record["metadata"]["category"] for record in records]).value_counts())

# 사람이 원문을 읽고 정한 평가 질문과 근거 원문 ID(evidence)입니다.
questions = read_json("lv1_questions.json")
question_by_id = {row["question_id"]: row for row in questions}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(questions)[["question_id", "question", "evidence"]])

# ====================================================================
# 8) 하이브리드 검색기
# 문서를 임베딩해 Chroma에 적재하고 BM25·Dense 검색기를 RRF로 연결합니다.
# ====================================================================
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day48_lv1_helpdesk", embedding_function=embedding_model)
vector_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = vector_store.add_documents(documents, ids=[doc.metadata["source_id"] for doc in documents])
print("day48_lv1_helpdesk 적재 수:", len(added_ids))

# 각 검색기가 최대 5개씩 찾으므로 합친 후보는 최대 10개입니다. 리랭킹으로 3개를 선택합니다.
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=5)
dense = vector_store.as_retriever(search_kwargs={"k": 5})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")

# ====================================================================
# 9) 리랭킹 구성요소와 모델
# 공식 리랭커 인터페이스를 가져오고 교안 01과 같은 Cross-Encoder를 준비합니다.
# ====================================================================
# 실행 시간과 공식 리랭커 인터페이스를 사용합니다.
from time import perf_counter
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# 첫 실행에는 공개 모델 파일을 내려받습니다. 노트북마다 한 번만 준비합니다.
# 실습은 CPU와 입력 상한 1024토큰을 사용합니다. 긴 입력은 잘릴 수 있습니다.
cross_encoder = HuggingFaceCrossEncoder(
    model_name="Qwen/Qwen3-Reranker-0.6B",
    model_kwargs={"device": "cpu", "max_length": 1024},
)

# ====================================================================
# 10) 순위 대조 함수
# rank_rows는 전후 순위를, evidence_rank는 근거 원문의 순위를 돌려줍니다.
# ====================================================================
def rank_rows(before, after):
    """원문 ID별 검색 순서와 후처리 순서를 대조하는 행 목록을 반환합니다."""
    # 같은 원문 ID로 비교해야 본문이 짧아져도 출처를 연결할 수 있습니다.
    after_ranks = {}
    for rank, doc in enumerate(after, start=1):
        source_id = doc.metadata["source_id"]
        after_ranks[source_id] = rank

    rows = []
    for rank, doc in enumerate(before, start=1):
        source_id = doc.metadata["source_id"]
        rows.append({"source_id": source_id, "before_rank": rank,
                     "after_rank": after_ranks.get(source_id)})
    return rows

def evidence_rank(documents, evidence_ids):
    """근거 원문이 목록에서 처음 나오는 순위를 돌려줍니다. 없으면 None입니다."""
    # 순위는 1부터 셉니다. 근거가 여러 개면 가장 앞에 나온 근거의 순위입니다.
    for rank, doc in enumerate(documents, start=1):
        if doc.metadata["source_id"] in evidence_ids:
            return rank
    return None


## 1. 리랭킹에 넘길 후보를 한 번 검색해 보관합니다

<strong>배경</strong>: 리랭킹 전후를 공정하게 비교하려면 같은 후보를 계속 써야 합니다. 후보를 한 번 검색해 저장하고, 리랭킹 없이 검색 순서대로 고른 상위 3개도 따로 남깁니다.

<strong>요구사항</strong>:

- <strong>`candidates`</strong>: 준비된 `hybrid`로 `question`을 한 번 검색한 결과 전체입니다. 리랭커에 넘길 1차 후보입니다.
- <strong>`initial_top`</strong>: `candidates`의 앞 3개를 담은 목록입니다. 리랭킹 없이 검색 순서대로 고른 최종 결과이며 3번에서 리랭킹 결과와 비교합니다.

<strong>확인 기준</strong>: `candidates`는 `Document` 5~10개입니다(BM25·Dense가 5개씩 찾은 결과의 합집합). `initial_top`은 3개입니다. 후보 수와 최종 개수 3은 서로 다른 설정입니다. 근거 원문의 순위는 실행 환경에 따라 달라질 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색은 한 번만 하고 최종 개수는 저장한 목록을 잘라서 정합니다.

세부구현:
1. hybrid의 invoke로 question을 검색해 candidates에 담습니다.
2. 슬라이싱으로 candidates의 앞 3개를 initial_top에 담습니다.
```

</details>


In [ ]:
# 1) 1~5번에서 함께 쓸 질문과, 사람이 원문을 읽고 정한 근거 원문 ID입니다(lv1_questions.json의 q01).
question = "해외 출장 중 호텔 와이파이에서 사내 VPN이 몇 분마다 끊겨요. 해외 접속 신청을 다시 해야 하나요?"
evidence_ids = {"help03"}

# ====================================================================
# [작성] 2) hybrid.invoke로 question을 한 번 검색해 결과 전체를 candidates에 담으세요.
# initial_top에는 candidates의 앞 3개를 슬라이싱으로 담습니다.
candidates = ...
initial_top = ...
# ====================================================================

# 3) 후보 수와 근거 원문의 위치를 확인합니다. 근거가 후보 밖이면 리랭킹도 찾지 못합니다.
print("후보 수:", len(candidates), "/ 최종으로 남길 수: 3")
print("후보 안에서 근거 원문의 순위:", evidence_rank(candidates, evidence_ids))
show_results(candidates)


In [ ]:
# [자가채점]
assert isinstance(candidates, list) and 5 <= len(candidates) <= 10 and all(isinstance(doc, Document) for doc in candidates), "hybrid.invoke(question)의 결과 전체를 candidates에 담으세요. 후보는 Document 5~10개입니다."
# 확인용으로 한 번 더 검색합니다(임베딩 요청 1회). 순서는 비교하지 않고 원문 ID 집합만 봅니다.
assert {doc.metadata["source_id"] for doc in hybrid.invoke(question)} == {doc.metadata["source_id"] for doc in candidates}, "candidates가 hybrid 검색 결과와 다릅니다. bm25나 dense 하나가 아니라 hybrid.invoke(question)의 결과 전체를 담으세요. 코드가 맞는데 이 메시지가 나오면 1번 셀을 한 번 더 실행하세요."
assert initial_top == candidates[:3], "initial_top에는 candidates의 앞 3개를 순서대로 담으세요."
print("✅ 1번 핵심 확인 완료!")


## 2. 질문과 후보를 한 쌍씩 묶어 관련성 점수를 구합니다

<strong>배경</strong>: Cross-Encoder는 질문과 문서 하나를 한 쌍으로 입력받아 관련성 점수를 냅니다. 입력 쌍을 준비하고 한 번에 점수를 받아 어떤 후보의 점수가 높은지 확인합니다.

<strong>요구사항</strong>:

- <strong>`pairs`</strong>: `candidates` 순서대로 `(question, 문서 본문)` 튜플을 담은 목록입니다. 문서 본문은 각 `Document`의 `page_content`입니다.
- <strong>`scores`</strong>: 준비된 `cross_encoder`의 `score`에 `pairs` 전체를 한 번에 전달한 결과입니다.

<strong>확인 기준</strong>: `pairs`와 `scores`의 길이는 `candidates`와 같습니다. 모든 튜플의 첫 항목은 같은 `question`입니다. 점수는 이 모델이 매긴 상대적 관련성이며 정답 확률이 아닙니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문은 고정하고 두 번째 항목만 후보마다 바꾼 뒤, 점수는 목록 전체로 한 번에 받습니다.

세부구현:
1. 리스트 컴프리헨션으로 candidates를 순서대로 돌며 (question, doc.page_content)를 만듭니다.
2. cross_encoder.score에 pairs를 전달합니다.
```

</details>


In [ ]:
# ====================================================================
# [작성] 1) candidates의 문서마다 (question, doc.page_content) 튜플을 만들어 pairs 목록에 담으세요.
# 리스트 컴프리헨션으로 만들면 candidates의 순서가 그대로 유지됩니다.
# [작성] 2) cross_encoder.score에 pairs 전체를 한 번에 전달해 scores에 담으세요.
pairs = ...
scores = ...
# ====================================================================

# 3) 점수는 pairs와 같은 순서입니다. 원문 ID·제목과 같은 위치끼리 묶어 표로 읽습니다.
score_rows = [
    {"source_id": doc.metadata["source_id"], "title": doc.metadata["title"], "score": float(score)}
    for doc, score in zip(candidates, scores)
]
# 점수가 높은 순서로 보여 주기만 합니다. 실제 재정렬은 3번의 공식 리랭커가 합니다.
display(pd.DataFrame(score_rows).sort_values("score", ascending=False))


In [ ]:
# [자가채점]
assert isinstance(pairs, list) and len(pairs) == len(candidates), "candidates의 문서마다 튜플 하나씩 담은 목록을 pairs에 만드세요."
assert all(pair == (question, doc.page_content) for pair, doc in zip(pairs, candidates)), "각 튜플은 (question, 같은 위치 후보의 page_content)입니다. 순서와 항목을 확인하세요."
assert len(scores) == len(candidates), "pairs 전체를 cross_encoder.score에 한 번 전달해 후보마다 점수 하나씩 받으세요."
assert len({round(float(score), 6) for score in scores}) > 1, "점수가 모두 같습니다. 점수를 직접 만들지 말고 cross_encoder.score(pairs)가 돌려준 값을 담으세요."
# 같은 쌍 목록을 한 번에 다시 점수화해 대조합니다(로컬 모델, 수 초). 같은 묶음이면 값도 같습니다.
assert all(abs(float(score) - float(expected)) < 1e-3 for score, expected in zip(scores, cross_encoder.score(pairs))), "scores가 cross_encoder.score(pairs)의 결과와 다릅니다. 모델이 돌려준 값을 순서 그대로 담으세요."
print("✅ 2번 핵심 확인 완료!")


## 3. 공식 리랭커로 같은 후보를 재정렬합니다

<strong>배경</strong>: `CrossEncoderReranker`는 점수 계산, 높은 순 정렬, 앞 `top_n`개 선택을 한 번에 처리합니다. 1번의 후보를 그대로 넣고 리랭킹 전 상위 3개와 무엇이 달라졌는지 비교합니다.

<strong>요구사항</strong>:

- <strong>`reranker`</strong>: 준비된 `cross_encoder`를 `model`로, `top_n=3`으로 만든 `CrossEncoderReranker`입니다. 모델을 새로 불러오지 않습니다.
- <strong>`reranked`</strong>: `reranker.compress_documents`에 1번의 `candidates`와 `question`을 이 순서로 넣은 결과를 `list`로 바꾼 목록입니다. 다시 검색하지 않습니다.

<strong>확인 기준</strong>: `reranked`는 3개이며 원문 본문과 메타데이터는 바뀌지 않습니다. 2번 표에서 점수가 높은 3개와 같은 원문입니다. 두 출력을 보고 어느 쪽이 질문에 답하는 원문을 앞에 두는지 읽으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 모델은 재사용하고 저장한 후보를 리랭커에 직접 넣습니다.

세부구현:
1. CrossEncoderReranker에 model과 top_n을 지정합니다.
2. compress_documents에 candidates, question 순서로 넣고 list로 바꿉니다.
```

</details>


In [ ]:
# 1) 모델 로딩을 뺀 재정렬 시간만 잽니다. 4번에서 후보가 적을 때와 비교합니다.
rerank_started = perf_counter()

# ====================================================================
# [작성] 2) CrossEncoderReranker에 model=cross_encoder, top_n=3을 지정해 reranker를 만드세요.
# [작성] 3) reranker.compress_documents(문서 목록, 질문)에 candidates와 question을 넣고,
#          결과를 list로 바꿔 reranked에 담으세요.
reranker = ...
reranked = ...
# ====================================================================

rerank_seconds = perf_counter() - rerank_started

# 4) 원문 ID별 전후 순위를 대조합니다. 최종 3개에서 빠진 후보는 after_rank가 비어 있습니다.
# dtype=object로 두면 빠진 후보의 순위가 NaN이 아니라 None으로 보입니다.
display(pd.DataFrame(rank_rows(candidates, reranked), dtype=object))
print("근거 원문의 순위(리랭킹 전 상위 3개):", evidence_rank(initial_top, evidence_ids))
print("근거 원문의 순위(리랭킹 후):", evidence_rank(reranked, evidence_ids))
print(f"재정렬 시간: {rerank_seconds:.1f}초 / 후보 {len(candidates)}개")
print("리랭킹 전 상위 3개")
show_results(initial_top)
print("리랭킹 후 상위 3개")
show_results(reranked)


In [ ]:
# [자가채점]
assert isinstance(reranker, CrossEncoderReranker) and reranker.model is cross_encoder, "모델을 새로 불러오지 말고 준비된 cross_encoder를 model로 넣어 CrossEncoderReranker를 만드세요."
assert reranker.top_n == 3, "reranker의 top_n을 3으로 설정하세요."
assert isinstance(reranked, list) and len(reranked) == 3, "compress_documents의 결과를 list로 바꿔 reranked에 담으세요. 결과는 3개입니다."
assert all(any(doc.page_content == original.page_content and doc.metadata == original.metadata for original in candidates) for doc in reranked), "리랭커는 후보 원문을 바꾸지 않고 고르기만 합니다. 1번의 candidates를 그대로 전달하세요."
score_order = sorted(zip(candidates, scores), key=lambda pair: pair[1], reverse=True)
assert [doc.metadata["source_id"] for doc in reranked] == [doc.metadata["source_id"] for doc, score in score_order[:3]], "2번 점수 상위 3개와 다릅니다. compress_documents에 1번의 candidates와 question을 넣었는지, 2번 scores가 cross_encoder.score(pairs)의 결과인지 확인하세요."
print("✅ 3번 핵심 확인 완료!")


## 4. 후보 밖의 근거는 리랭킹으로 찾을 수 없습니다

<strong>배경</strong>: 처리 시간을 줄이려고 임베딩 없이 BM25 상위 2개만 리랭커에 넘기는 방안을 검토합니다. 리랭커는 받은 후보 안에서만 고르므로 후보를 너무 좁히면 근거를 놓칠 수 있습니다.

<strong>요구사항</strong>:

- <strong>`narrow_candidates`</strong>: 준비된 `narrow_bm25`로 `question`을 검색한 결과입니다.
- <strong>`narrow_reranked`</strong>: 3번 `reranker`의 `compress_documents`로 `narrow_candidates`를 재정렬한 결과를 `list`로 바꾼 목록입니다.

<strong>확인 기준</strong>: `narrow_candidates`는 2개이고 `narrow_reranked`도 그 2개 안의 원문만 담습니다. 근거 원문이 후보에 없으면 리랭킹 결과에도 없습니다. 이것이 이 문항의 관찰 대상이며 오답이 아닙니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 리랭커에 후보 수만 다르게 넣고 근거가 남았는지 비교합니다.

세부구현:
1. narrow_bm25의 invoke로 question을 검색합니다.
2. reranker의 compress_documents에 narrow_candidates와 question을 넣고 list로 바꿉니다.
```

</details>


In [ ]:
# 1) 임베딩 없이 BM25 상위 2개만 후보로 쓰는 검색기입니다. 먼저 BM25 상위 5개 안에서 근거가 몇 위인지 봅니다.
print("BM25 상위 5개 안에서 근거 원문의 순위:", evidence_rank(bm25.invoke(question), evidence_ids))
narrow_bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=2)
narrow_started = perf_counter()

# ====================================================================
# [작성] 2) narrow_bm25.invoke로 question을 검색해 narrow_candidates에 담으세요.
# [작성] 3) 3번 reranker의 compress_documents로 narrow_candidates를 재정렬하고 list로 바꿔 narrow_reranked에 담으세요.
narrow_candidates = ...
narrow_reranked = ...
# ====================================================================

narrow_seconds = perf_counter() - narrow_started

# 4) 두 설정의 후보 수, 근거 포함 여부, 재정렬 시간을 한 표로 비교합니다.
# 4번의 시간에는 BM25 검색 시간도 조금 들어 있습니다.
display(pd.DataFrame([
    {"setting": "하이브리드 후보", "candidate_count": len(candidates),
     "evidence_in_candidates": evidence_rank(candidates, evidence_ids) is not None,
     "evidence_rank_after": evidence_rank(reranked, evidence_ids), "seconds": round(rerank_seconds, 1)},
    {"setting": "BM25 상위 2개", "candidate_count": len(narrow_candidates),
     "evidence_in_candidates": evidence_rank(narrow_candidates, evidence_ids) is not None,
     "evidence_rank_after": evidence_rank(narrow_reranked, evidence_ids), "seconds": round(narrow_seconds, 1)},
], dtype=object))
show_results(narrow_reranked)


In [ ]:
# [자가채점]
assert isinstance(narrow_candidates, list) and [doc.metadata["source_id"] for doc in narrow_candidates] == [doc.metadata["source_id"] for doc in narrow_bm25.invoke(question)], "narrow_bm25.invoke(question)의 결과를 순서 그대로 narrow_candidates에 담으세요."
assert isinstance(narrow_reranked, list) and len(narrow_reranked) == len(narrow_candidates), "narrow_reranked는 narrow_candidates(2개)를 reranker로 재정렬한 결과입니다. 1번의 candidates를 넣지 않았는지 확인하세요. top_n=3보다 후보가 적으면 후보 수만큼 나옵니다."
assert [doc.metadata["source_id"] for doc in narrow_reranked] == [doc.metadata["source_id"] for doc in reranker.compress_documents(narrow_candidates, question)], "narrow_reranked에는 reranker.compress_documents(narrow_candidates, question)의 결과를 담으세요."
print("✅ 4번 핵심 확인 완료!")


5번에서 쓸 ColBERT 방식 모델입니다. 아래 셀은 실행만 하세요. 교안 01과 같은 `BAAI/bge-m3` 모델을 토큰별 벡터 설정으로 불러옵니다(`FlagEmbedding==1.4.2`, 가중치 약 2.3GB).


In [ ]:
# ====== ColBERT 방식의 토큰 벡터 모델 준비 ======
from FlagEmbedding import BGEM3FlagModel

# 교안 01과 같은 설정입니다. 토큰별 벡터만 받고 문서 하나짜리 벡터와 희소 벡터는 끕니다.
colbert_model = BGEM3FlagModel(
    "BAAI/bge-m3",
    devices="cpu", use_fp16=False, batch_size=2,
    query_max_length=1024, passage_max_length=1024,
    return_dense=False, return_sparse=False, return_colbert_vecs=True,
)


## 5. ColBERT 방식으로 같은 후보를 재정렬합니다

<strong>배경</strong>: Cross-Encoder는 질문·문서 쌍을 한 번에 입력받았습니다. ColBERT 방식은 질문과 문서를 각각 토큰별 벡터로 만든 뒤 질문 토큰마다 가장 잘 맞는 문서 토큰을 찾아 점수를 냅니다. 1번의 후보 전체를 이 방식으로 점수화하고 두 방식이 고른 원문을 비교합니다.

<strong>요구사항</strong>:

- <strong>`query_vectors`</strong>: `colbert_model.encode_queries`에 `[question]`을 넣은 결과의 `"colbert_vecs"`에서 첫 번째 항목입니다(질문 하나의 토큰별 벡터).
- <strong>`document_vectors`</strong>: `colbert_model.encode_corpus`에 `candidates`의 본문 목록을 넣은 결과의 `"colbert_vecs"`입니다(후보마다 토큰별 벡터).
- <strong>`colbert_scores`</strong>: `document_vectors`를 순서대로 돌며 `colbert_model.colbert_score(query_vectors, 그 후보의 벡터)`를 `float`으로 바꿔 담은 목록입니다.

<strong>확인 기준</strong>: `colbert_scores`는 후보 수만큼 있고 `candidates`와 같은 순서입니다. `query_vectors`는 (질문 토큰 수, 벡터 차원) 모양입니다. 두 방식의 결과가 같아도 정상입니다. 점수 척도가 다르므로 Qwen 점수와 ColBERT 점수를 서로 비교하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 벡터와 후보 벡터를 따로 만든 뒤, 후보마다 같은 질문 벡터와 점수를 냅니다.

세부구현:
1. encode_queries의 반환 딕셔너리에서 colbert_vecs를 꺼내 첫 항목을 고릅니다.
2. 리스트 컴프리헨션으로 후보 본문 목록을 만들어 encode_corpus에 넣습니다.
3. 리스트 컴프리헨션으로 document_vectors를 돌며 colbert_score를 부르고 float으로 바꿉니다.
```

</details>


In [ ]:
# ====================================================================
# [작성] 1) colbert_model.encode_queries([question])["colbert_vecs"]의 첫 항목을 query_vectors에 담으세요.
#          colbert_vecs는 질문마다 배열 하나를 담은 목록이라 [0]으로 첫 질문의 배열을 꺼냅니다.
# [작성] 2) candidates의 page_content 목록을 encode_corpus에 넣고 결과의 "colbert_vecs"를 document_vectors에 담으세요.
# [작성] 3) document_vectors를 순서대로 돌며 colbert_score(query_vectors, 후보 벡터)를 float으로 바꿔 colbert_scores에 담으세요.
query_vectors = ...
document_vectors = ...
colbert_scores = ...
# ====================================================================

# 4) 점수가 높은 3개 후보를 고릅니다. 정렬과 선택은 교안 01과 같은 제공 코드입니다.
colbert_order = sorted(range(len(candidates)), key=lambda index: colbert_scores[index], reverse=True)
colbert_reranked = [candidates[index] for index in colbert_order[:3]]
print("질문 벡터 모양(토큰 수, 벡터 차원):", query_vectors.shape)

# 5) 같은 후보에서 두 방식이 고른 원문 ID와 근거 순위를 나란히 봅니다.
display(pd.DataFrame([
    {"method": "Qwen Cross-Encoder", "source_ids": [doc.metadata["source_id"] for doc in reranked],
     "evidence_rank": evidence_rank(reranked, evidence_ids)},
    {"method": "BGE-M3 ColBERT", "source_ids": [doc.metadata["source_id"] for doc in colbert_reranked],
     "evidence_rank": evidence_rank(colbert_reranked, evidence_ids)},
], dtype=object))


In [ ]:
# [자가채점]
assert not isinstance(query_vectors, list) and query_vectors.ndim == 2, "query_vectors는 encode_queries([question])의 colbert_vecs에서 첫 항목([0])입니다. 목록 전체를 담지 않았는지 확인하세요."
assert len(document_vectors) == len(candidates), "document_vectors는 candidates의 본문 목록을 encode_corpus에 넣은 결과의 colbert_vecs입니다. 후보마다 하나씩 있어야 합니다."
# 확인용으로 같은 입력을 다시 인코딩합니다(로컬 모델, 수 초). 토큰 수가 같은지로 입력을 대조합니다.
assert query_vectors.shape == colbert_model.encode_queries([question])["colbert_vecs"][0].shape, "query_vectors가 question의 토큰 벡터와 다릅니다. 1번의 question을 encode_queries에 넣었는지 확인하세요."
expected_shapes = [vectors.shape for vectors in colbert_model.encode_corpus([doc.page_content for doc in candidates])["colbert_vecs"]]
assert [vectors.shape for vectors in document_vectors] == expected_shapes, "document_vectors가 candidates의 본문과 맞지 않습니다. candidates 순서대로 page_content 목록을 encode_corpus에 넣었는지 확인하세요."
assert isinstance(colbert_scores, list) and len(colbert_scores) == len(candidates) and all(type(score) is float for score in colbert_scores), "colbert_scores는 후보마다 float 점수 하나씩 담은 목록입니다."
assert all(abs(score - float(colbert_model.colbert_score(query_vectors, vectors))) < 1e-6 for score, vectors in zip(colbert_scores, document_vectors)), "colbert_scores에는 같은 위치의 document_vectors와 query_vectors로 구한 colbert_score를 담으세요."
print("✅ 5번 핵심 확인 완료!")


## 6. 검색과 리랭킹을 하나의 검색기로 연결합니다

<strong>배경</strong>: 헬프데스크 챗봇에 넣으려면 새 질문마다 검색과 리랭킹을 따로 부르지 않고 `invoke` 한 번으로 처리해야 합니다. `ContextualCompressionRetriever`로 두 단계를 연결하고 새 질문에 적용합니다.

<strong>요구사항</strong>:

- <strong>`ranked_retriever`</strong>: `base_retriever`에 `hybrid`, `base_compressor`에 3번 `reranker`를 넣은 `ContextualCompressionRetriever`입니다.
- <strong>`new_results`</strong>: `ranked_retriever`로 준비된 `new_question`을 한 번 `invoke`한 결과입니다.

<strong>확인 기준</strong>: `new_results`는 3개입니다. 이 검색기는 호출할 때마다 1차 검색부터 다시 하므로, 같은 후보로 모델을 비교하는 7번에서는 저장한 후보를 직접 리랭커에 넣습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 후보를 찾는 객체와 받은 후보를 처리하는 객체를 역할에 맞는 인자에 연결합니다.

세부구현:
1. ContextualCompressionRetriever의 base_retriever와 base_compressor에 두 객체를 지정합니다.
2. 만든 검색기의 invoke에 new_question을 전달합니다.
```

</details>


In [ ]:
# 1) 새로 들어온 질문입니다(lv1_questions.json의 q07).
new_question = "회의실 TV에 노트북 화면이 안 나와요. 무엇을 확인해야 하나요?"

# ====================================================================
# [작성] 2) ContextualCompressionRetriever에 base_retriever=hybrid, base_compressor=reranker를 지정해
#          ranked_retriever를 만드세요. 두 자리를 바꾸면 BaseDocumentCompressor가 필요하다는 ValidationError가 납니다.
# [작성] 3) ranked_retriever.invoke로 new_question을 검색해 new_results에 담으세요.
ranked_retriever = ...
new_results = ...
# ====================================================================

# 4) 검색·리랭킹을 한 번에 거친 결과입니다. 근거 원문(help34)의 위치를 확인합니다.
print("근거 원문의 순위:", evidence_rank(new_results, set(question_by_id["q07"]["evidence"])))
show_results(new_results)


In [ ]:
# [자가채점]
assert isinstance(ranked_retriever, ContextualCompressionRetriever), "ContextualCompressionRetriever로 ranked_retriever를 만드세요."
assert ranked_retriever.base_retriever is hybrid and ranked_retriever.base_compressor is reranker, "base_retriever에는 hybrid, base_compressor에는 3번 reranker를 연결하세요."
assert isinstance(new_results, list) and len(new_results) == 3 and all(isinstance(doc, Document) for doc in new_results), "ranked_retriever.invoke(new_question)의 결과(Document 3개)를 new_results에 담으세요."
assert new_results != reranked, "new_results가 1번 질문의 결과와 같습니다. question이 아니라 new_question으로 invoke했는지 확인하세요."
print("✅ 6번 핵심 확인 완료!")


## 7. 모델만 바꿔 여러 질문의 같은 후보로 비교합니다

<strong>배경</strong>: 리랭커를 고를 때는 질문 하나가 아니라 우리 문서의 평가 질문 여러 개로, 같은 후보와 같은 `top_n`에서 비교합니다. Qwen과 BGE 리랭커를 평가 질문 5개에 적용해 근거 순위와 처리 시간을 기록합니다.

<strong>요구사항</strong>:

- <strong>`bge_encoder`</strong>: `HuggingFaceCrossEncoder`로 `model_name="BAAI/bge-reranker-v2-m3"` 모델을 준비합니다. `model_kwargs`는 Qwen과 같은 `{"device": "cpu", "max_length": 1024}`입니다.
- <strong>`bge_reranker`</strong>: `bge_encoder`를 `model`로, `top_n=3`으로 만든 `CrossEncoderReranker`입니다.

<strong>확인 기준</strong>: 비교 표는 제공 코드가 만듭니다. `no_rerank`·`qwen`·`bge` 칸은 최종 3개 안에서 근거 원문의 순위이고, `None`은 최종 3개에 근거가 없다는 뜻입니다. 리랭킹 전에도 근거가 1위인 질문이 많으면 두 모델의 차이는 주로 처리 시간(`qwen_seconds`·`bge_seconds`)에서 드러납니다. 순위와 시간은 실행 환경마다 달라집니다. 결과는 `output/lv1_reranker_check.json`에 저장됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델 이름만 바꿔 같은 방식으로 Cross-Encoder와 리랭커를 만듭니다.

세부구현:
1. 준비된 cross_encoder와 같은 모양으로 HuggingFaceCrossEncoder를 만들고 model_name만 바꿉니다.
2. CrossEncoderReranker에 bge_encoder와 top_n을 지정합니다.
```

</details>


In [ ]:
# 1) 5번 뒤로 쓰지 않는 ColBERT 모델을 메모리에서 내립니다. 5번을 다시 풀려면 5번 앞의 ColBERT 모델 준비 셀을 다시 실행하세요.
colbert_model = None

# 2) 평가 질문 5개(q02~q06)의 후보를 한 번만 검색해 두 모델에 똑같이 넣습니다.
check_questions = [row for row in questions if row["question_id"] in {"q02", "q03", "q04", "q05", "q06"}]
check_candidates = hybrid.batch([row["question"] for row in check_questions])

# ====================================================================
# [작성] 3) HuggingFaceCrossEncoder에 model_name="BAAI/bge-reranker-v2-m3"와
#          model_kwargs={"device": "cpu", "max_length": 1024}를 지정해 bge_encoder를 만드세요.
# [작성] 4) bge_encoder를 model로, top_n=3으로 만든 CrossEncoderReranker를 bge_reranker에 담으세요.
bge_encoder = ...
bge_reranker = ...
# ====================================================================

# 5) 같은 후보·같은 top_n으로 두 리랭커를 실행해 근거 순위와 시간을 기록합니다. 이 블록은 그대로 실행하세요.
compare_rows = []
for row, found in zip(check_questions, check_candidates):
    evidence = set(row["evidence"])
    result = {"question_id": row["question_id"], "candidate_count": len(found),
              "no_rerank": evidence_rank(found[:3], evidence)}
    for name, model_reranker in {"qwen": reranker, "bge": bge_reranker}.items():
        started = perf_counter()
        selected = list(model_reranker.compress_documents(found, row["question"]))
        result[name] = evidence_rank(selected, evidence)
        result[name + "_seconds"] = round(perf_counter() - started, 2)
    compare_rows.append(result)
display(pd.DataFrame(compare_rows, dtype=object))
# 근거가 1위인 질문 수를 방식별로 셉니다. 차이가 작으면 처리 시간이 선택을 가릅니다.
print("근거가 1위인 질문 수:", {name: sum(row[name] == 1 for row in compare_rows) for name in ["no_rerank", "qwen", "bge"]})
print("질문당 평균 처리 시간(초):", {name: round(sum(row[name + "_seconds"] for row in compare_rows) / len(compare_rows), 2) for name in ["qwen", "bge"]})

# 6) 비교 조건과 결과를 함께 저장합니다. 8번 서술의 근거로 씁니다.
save_json("lv1_reranker_check.json", {
    "models": {"qwen": cross_encoder.model_name, "bge": bge_encoder.model_name},
    "top_n": 3,
    "candidate_ids": {row["question_id"]: [doc.metadata["source_id"] for doc in found]
                      for row, found in zip(check_questions, check_candidates)},
    "results": compare_rows,
})


In [ ]:
# [자가채점]
assert isinstance(bge_encoder, HuggingFaceCrossEncoder) and bge_encoder.model_name == "BAAI/bge-reranker-v2-m3", "HuggingFaceCrossEncoder의 model_name을 \"BAAI/bge-reranker-v2-m3\"로 지정하세요."
assert bge_encoder.model_kwargs == {"device": "cpu", "max_length": 1024}, "model_kwargs를 Qwen과 같은 {\"device\": \"cpu\", \"max_length\": 1024}로 지정하세요. 비교 조건을 맞춰야 합니다."
assert isinstance(bge_reranker, CrossEncoderReranker) and bge_reranker.model is bge_encoder and bge_reranker.top_n == 3, "bge_encoder를 model로, top_n=3으로 CrossEncoderReranker를 만드세요."
print("✅ 7번 핵심 확인 완료!")


## 8. 운영 조건에 맞는 리랭커를 고릅니다

<strong>배경</strong>: 리랭커는 품질뿐 아니라 문서를 밖으로 보낼 수 있는지, 어떤 장비에서 얼마나 빨리 응답해야 하는지에 따라 고릅니다. 7번 결과와 교안 01 6절의 모델 비교 표를 근거로 두 상황의 선택을 제안합니다.

<strong>요구사항</strong>:

- <strong>방식 비교</strong>: 도움말 41개를 밤사이 미리 처리해 둔다고 할 때, Dense(Bi-Encoder)·Cross-Encoder·ColBERT 방식이 각각 무엇을 저장해 두고 질문 하나가 들어오면 무엇을 계산하는지 방식마다 한 문장씩 쓰세요.
- <strong>상황 A</strong>: 헬프데스크 문서를 외부로 보낼 수 없고, GPU가 없는 사내 서버에서 질문 하나를 몇 초 안에 처리해야 합니다. 검토할 모델과 먼저 확인할 것 두 가지를 쓰세요. 리랭커만이 아니라 검색부터 리랭킹까지 전체 흐름에서 외부로 나가는 요청이 있는지도 보세요.
- <strong>상황 B</strong>: 외부 전송이 승인되었고, 로컬에서 모델을 운영할 서버가 없습니다. 검토할 모델과 확인할 것 두 가지를 쓰세요.
- <strong>공통</strong>: 7번처럼 평가 질문 5개로 비교한 결과만으로 결론을 내리기 어려운 이유를 한 문장으로 쓰세요.

<strong>확인 기준</strong>: 방식 비교는 방식마다 한 문장, 상황은 2~4문장으로 씁니다. 모델 이름만 고르지 말고 실행 방식·처리 시간·자원·비용·호출 한도, Hugging Face 모델 카드의 지원 언어·파라미터 수·라이선스 가운데 근거를 듭니다. 코드는 작성하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 외부 전송 가능 여부로 먼저 후보를 가르고, 실행 자원과 처리 시간으로 좁힙니다.

세부구현:
1. 교안 01 2절 표에서 세 방식이 문서 쪽에 미리 준비하는 것을 확인합니다.
2. 7번 표의 qwen_seconds와 bge_seconds, 근거 순위를 함께 봅니다.
3. 교안 01 6절 표에서 로컬 모델과 API 모델의 실행 방식을 확인하고, 이 과제의 임베딩이 어디서 계산되는지 떠올립니다.
4. 평가 질문이 5개뿐이라는 점과 점수 척도가 모델마다 다르다는 점을 떠올립니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
